# Definitive AI Training Analysis

This notebook is the canonical review of the deck-builder experiments. It is currently configured for the multi-generation `final-meta-evolution` experiment, checks data integrity before interpretation, and distinguishes observed results from proxies.

**Primary questions**

1. Did every run produce complete, balanced, reproducible evaluation data?
2. Did the outer deck builder improve and retain sufficient diversity?
3. How does controller behavior change with inner generation?
4. Do active-ability cards become more valuable as controllers train?
5. Which card and deck signals replicate across independent seeds?

Historical runs do not record individual ability-use decisions, so their active-card analysis is a performance proxy. The inner-ceiling protocols add direct play and ability-use summaries plus cross-generation play.

## 1. Environment and configuration

Run this notebook from anywhere inside the repository. Database connections are read-only. The first aggregation of the large `pc_match` table can take several minutes; unchanged reruns load the result from `.analysis_cache/`.

**Run-selection checklist**

- [x] Select the final experiment by its stable `final-meta-evolution` label and four pre-registered seeds.
- [x] Exclude failed runs automatically and show every admitted run before large queries.
- [x] Keep incomplete-run protection enabled until every selected seed completes.
- [ ] Use explicit IDs only when intentionally reviewing a subset of one labelled experiment.
- [ ] Restart the kernel and run all cells after changing the selection.

In [ ]:
import hashlib
import json
import math
import sqlite3
import sys
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
px.defaults.template = 'plotly_white'

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'train.db').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'train.db').exists():
    raise FileNotFoundError('Could not locate train.db from the current directory')

TRAIN_DB = PROJECT_ROOT / 'train.db'
CARDS_DB = PROJECT_ROOT / 'utils' / 'cards.db'
PRIMARY_LABEL = 'replicated-final'
SENSITIVITY_LABEL = 'post-replication'
CEILING_LABELS = ('inner-ceiling-50', 'inner-ceiling-65')
FINAL_COLLECTION_LABEL = 'final-deck-discovery'
META_EVOLUTION_LABEL = 'final-meta-evolution'
# Keep empty when reporting only the selected experiment. Add (9, 10) only for
# explicitly labelled historical comparisons.
LEGACY_REFERENCE_IDS = ()

# Selection precedence: explicit IDs, explicit labels, then all known labels.
# The final thesis analysis intentionally admits only this experiment label.
ANALYSIS_RUN_LABELS = (META_EVOLUTION_LABEL,)
ANALYSIS_EXPECTED_SEEDS = (667615478, 2069633686, 915632071, 274633778)
ANALYSIS_EXPECTED_CONFIG = {
    'configured_outer_generations': 10,
    'configured_inner_generations': 50,
    'holdout_seed_pairs': 20,
}
ANALYSIS_TRAINING_IDS = ()
CHECKPOINTS = (1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65)
ALLOW_INCOMPLETE_EXPERIMENTS = False  # Keep false while training is running.
USE_AGGREGATE_CACHE = True
SHOW_INDIVIDUAL_RUNS = True  # Preserve seed-level traces beneath summaries.
DIVERSITY_DISTANCE_SCALE = 1.0  # One unit equals one card replacement.

print(f'Project: {PROJECT_ROOT}')
print(f'Training DB: {TRAIN_DB} ({TRAIN_DB.stat().st_size / 2**30:.2f} GiB)')

In [ ]:
@contextmanager
def db_connection():
    conn = sqlite3.connect(f'file:{TRAIN_DB}?mode=ro', uri=True, timeout=120)
    conn.execute('PRAGMA query_only = ON')
    conn.execute('PRAGMA temp_store = MEMORY')
    conn.execute('PRAGMA cache_size = -262144')
    conn.execute('ATTACH DATABASE ? AS cardsdb', (str(CARDS_DB),))
    try:
        yield conn
    finally:
        conn.close()

def read_sql(query, params=None):
    with db_connection() as conn:
        return pd.read_sql_query(query, conn, params=params)

def sql_in(values):
    values = tuple(int(value) for value in values)
    if not values:
        raise ValueError('At least one run ID is required')
    return ','.join('?' for _ in values), values

def weighted_mean(frame, value, weight='sample_count'):
    valid = frame[[value, weight]].dropna()
    total_weight = valid[weight].sum()
    return np.nan if total_weight == 0 else np.average(valid[value], weights=valid[weight])

def safe_corr(left, right):
    valid = pd.concat([left, right], axis=1).dropna()
    if len(valid) < 2 or valid.iloc[:, 0].nunique() < 2 or valid.iloc[:, 1].nunique() < 2:
        return np.nan
    return valid.iloc[:, 0].corr(valid.iloc[:, 1])

def wilson_interval(wins, games, z=1.96):
    if games <= 0:
        return np.nan, np.nan
    p = wins / games
    denominator = 1 + z * z / games
    center = (p + z * z / (2 * games)) / denominator
    margin = z * math.sqrt(p * (1 - p) / games + z * z / (4 * games * games)) / denominator
    return center - margin, center + margin

## 2. Run catalogue

Runs enter analysis through the selectors above. The table records why each candidate is admitted or excluded, its human-readable purpose, completion state, and effective configuration before any expensive match query runs.

In [ ]:
run_catalog = read_sql('''
    SELECT
        t.id AS training_id,
        t.start_date,
        t.end_date,
        t.description,
        t.metadata,
        COUNT(DISTINCT g.guid) AS genomes,
        COALESCE(MAX(g.gen), 0) AS observed_outer_generations,
        COUNT(DISTINCT m.guid) AS outer_matches,
        SUM(CASE WHEN m.result IS NULL THEN 1 ELSE 0 END) AS unfinished_outer_matches
    FROM training t
    LEFT JOIN genome g ON g.training_id = t.id
    LEFT JOIN match_dbt m ON m.genome_1 = g.guid
    GROUP BY t.id
    ORDER BY t.id
''')

def parse_metadata(value):
    if not value:
        return {}
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {}

run_catalog['metadata_dict'] = run_catalog['metadata'].map(parse_metadata)
for column, path in {
    'label': ('settings', 'run_label'),
    'seed': ('settings', 'seed'),
    'configured_outer_generations': ('settings', 'outer_generations'),
    'configured_inner_generations': ('settings', 'evaluator_generations'),
    'holdout_seed_pairs': ('settings', 'holdout_seed_pairs'),
    'outer_workers': ('settings', 'outer_workers'),
    'inner_workers': ('settings', 'evaluator_workers'),
    'git_dirty': ('git', 'dirty'),
    'termination_reason': ('termination_reason',),
}.items():
    run_catalog[column] = run_catalog['metadata_dict'].map(
        lambda metadata: metadata.get(path[0], {}).get(path[1])
        if len(path) == 2 else metadata.get(path[0])
    )

run_catalog['description'] = run_catalog.apply(
    lambda row: row['description'] or row['metadata_dict'].get('description')
    or row['metadata_dict'].get('settings', {}).get('description'), axis=1
)

failed_run_mask = run_catalog['termination_reason'].fillna('').str.startswith('failed:')
discovery_catalog = run_catalog.loc[~failed_run_mask]
excluded_failed_ids = tuple(run_catalog.loc[failed_run_mask, 'training_id'])

label_primary = tuple(discovery_catalog.loc[discovery_catalog['label'] == PRIMARY_LABEL, 'training_id'])
label_sensitivity = tuple(discovery_catalog.loc[discovery_catalog['label'] == SENSITIVITY_LABEL, 'training_id'])
label_convergence = tuple(discovery_catalog.loc[discovery_catalog['label'].isin(CEILING_LABELS), 'training_id'])
label_collection = tuple(discovery_catalog.loc[discovery_catalog['label'] == FINAL_COLLECTION_LABEL, 'training_id'])
label_meta_evolution = tuple(discovery_catalog.loc[discovery_catalog['label'] == META_EVOLUTION_LABEL, 'training_id'])
auto_discovered_ids = label_primary + label_sensitivity + label_convergence + label_collection + label_meta_evolution
label_candidate_mask = discovery_catalog['label'].isin(ANALYSIS_RUN_LABELS)
if ANALYSIS_EXPECTED_SEEDS:
    label_candidate_mask &= discovery_catalog['seed'].isin(ANALYSIS_EXPECTED_SEEDS)
label_selected_ids = tuple(discovery_catalog.loc[label_candidate_mask, 'training_id'])
if ANALYSIS_TRAINING_IDS:
    selection_mode = 'explicit training IDs'
    EXPERIMENT_IDS = tuple(dict.fromkeys(ANALYSIS_TRAINING_IDS))
elif ANALYSIS_RUN_LABELS:
    selection_mode = 'run labels'
    EXPERIMENT_IDS = tuple(dict.fromkeys(label_selected_ids))
else:
    selection_mode = 'automatic discovery of all known experiment labels'
    EXPERIMENT_IDS = tuple(dict.fromkeys(auto_discovered_ids))

catalog_id_set = set(run_catalog.training_id)
missing_selected_ids = tuple(run_id for run_id in EXPERIMENT_IDS if run_id not in catalog_id_set)
if missing_selected_ids:
    raise RuntimeError(f'Selected training IDs do not exist: {missing_selected_ids}')
available_labels = set(discovery_catalog['label'].dropna())
missing_selected_labels = tuple(
    label for label in ANALYSIS_RUN_LABELS if label not in available_labels
)
if not ANALYSIS_TRAINING_IDS and missing_selected_labels:
    raise RuntimeError(
        f'No non-failed runs exist for selected labels: {missing_selected_labels}. '
        'Start the experiment or check its --run-label value.'
    )
if not ANALYSIS_TRAINING_IDS and ANALYSIS_EXPECTED_SEEDS:
    selected_seed_counts = discovery_catalog.loc[
        discovery_catalog.training_id.isin(EXPERIMENT_IDS), 'seed'
    ].value_counts()
    missing_seeds = tuple(
        seed for seed in ANALYSIS_EXPECTED_SEEDS if selected_seed_counts.get(seed, 0) == 0
    )
    duplicate_seeds = tuple(
        int(seed) for seed, count in selected_seed_counts.items() if count > 1
    )
    if missing_seeds:
        raise RuntimeError(
            f'The selected experiment is missing pre-registered seeds: {missing_seeds}. '
            'Wait for the complete four-seed command before analysis.'
        )
    if duplicate_seeds:
        raise RuntimeError(
            f'Multiple runs use the selected label and seeds {duplicate_seeds}. '
            'Set ANALYSIS_TRAINING_IDS explicitly to choose one experiment instance.'
        )
    selected_protocol = discovery_catalog.loc[
        discovery_catalog.training_id.isin(EXPERIMENT_IDS)
    ]
    protocol_errors = []
    for column, expected_value in ANALYSIS_EXPECTED_CONFIG.items():
        invalid_ids = tuple(selected_protocol.loc[
            selected_protocol[column] != expected_value, 'training_id'
        ])
        if invalid_ids:
            protocol_errors.append(f'{column} != {expected_value} in {invalid_ids}')
    if protocol_errors:
        raise RuntimeError('Selected runs do not match the final protocol: ' + '; '.join(protocol_errors))

selected_id_set = set(EXPERIMENT_IDS)
discovered_primary = tuple(run_id for run_id in label_primary if run_id in selected_id_set)
discovered_sensitivity = tuple(run_id for run_id in label_sensitivity if run_id in selected_id_set)
discovered_convergence = tuple(run_id for run_id in label_convergence if run_id in selected_id_set)
discovered_collection = tuple(run_id for run_id in label_collection if run_id in selected_id_set)
discovered_meta_evolution = tuple(run_id for run_id in label_meta_evolution if run_id in selected_id_set)
REFERENCE_IDS = tuple(run_id for run_id in LEGACY_REFERENCE_IDS if run_id in set(run_catalog['training_id']))
QUERY_IDS = tuple(dict.fromkeys(EXPERIMENT_IDS + REFERENCE_IDS))

if not EXPERIMENT_IDS:
    raise RuntimeError('No experiment runs found; set ANALYSIS_RUN_LABELS or ANALYSIS_TRAINING_IDS')

catalog_columns = [
    'training_id', 'analysis_admission', 'label', 'description', 'seed',
    'start_date', 'end_date',
    'configured_outer_generations', 'observed_outer_generations',
    'configured_inner_generations', 'holdout_seed_pairs', 'genomes',
    'outer_matches', 'unfinished_outer_matches', 'git_dirty', 'termination_reason',
]
run_catalog['analysis_admission'] = np.select(
    [
        failed_run_mask,
        run_catalog.training_id.isin(EXPERIMENT_IDS),
        run_catalog.training_id.isin(REFERENCE_IDS),
    ],
    ['excluded: failed', 'admitted: experiment', 'admitted: reference'],
    default='not selected',
)
candidate_mask = (
    run_catalog['label'].isin(ANALYSIS_RUN_LABELS)
    | run_catalog['training_id'].isin(QUERY_IDS)
)
display(run_catalog.loc[candidate_mask, catalog_columns])
print('Selection mode:', selection_mode)
print('Requested labels:', ANALYSIS_RUN_LABELS or 'automatic')
print('Expected seeds:', ANALYSIS_EXPECTED_SEEDS or 'not constrained')
print('Admitted experiment runs:', EXPERIMENT_IDS)
print('Primary:', discovered_primary)
print('Sensitivity:', discovered_sensitivity or 'not present yet')
print('Convergence:', discovered_convergence or 'not present yet')
print('Final collection:', discovered_collection or 'not present yet')
print('Meta evolution:', discovered_meta_evolution or 'not present yet')
print('Legacy references:', REFERENCE_IDS)
print('Excluded failed runs (all labels):', excluded_failed_ids or 'none')

## 3. Outer-loop integrity and composition-aware diversity

Exact signatures are retained only as a reference. The primary diversity measure represents each deck as a card-count vector, defines one unit of distance as one card replacement, and discounts similar decks with an exponential similarity kernel. `one_change_archetypes` also joins decks connected by at most one card change; it is intentionally a coarse secondary view. Mean outer fitness is omitted because pairwise scoring is zero-sum; fitness spread measures population differentiation instead.

In [ ]:
placeholders, query_ids = sql_in(QUERY_IDS)
genomes = read_sql(f'''
    SELECT
        g.training_id,
        g.guid,
        g.gen AS outer_gen,
        g.deck_id,
        g.score,
        d.signature,
        d.name AS deck_name
    FROM genome g
    LEFT JOIN ai_training_decks d ON d.id = g.deck_id
    WHERE g.training_id IN ({placeholders})
    ORDER BY g.training_id, g.gen, g.guid
''', query_ids)

deck_cards = read_sql(f'''
    SELECT d.training_id, dl.deck_id, dl.card_id, COUNT(*) AS copies
    FROM ai_training_decks d
    JOIN ai_training_deck_lines dl ON dl.deck_id = d.id
    WHERE d.training_id IN ({placeholders})
    GROUP BY d.training_id, dl.deck_id, dl.card_id
''', query_ids)
deck_vectors = deck_cards.pivot_table(
    index='deck_id', columns='card_id', values='copies', fill_value=0
)

def composition_diversity(group):
    deck_ids = list(group.deck_id)
    vectors = deck_vectors.reindex(deck_ids, fill_value=0).to_numpy(dtype=float)
    count = len(vectors)
    if count == 0:
        return pd.Series(dtype=float)
    distances = np.abs(vectors[:, None, :] - vectors[None, :, :]).sum(axis=2) / 2
    similarities = np.exp(-distances / DIVERSITY_DISTANCE_SCALE)
    effective_decks = count * count / similarities.sum()
    nearest = distances.copy()
    np.fill_diagonal(nearest, np.inf)

    parents = list(range(count))
    def find(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index
    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parents[right_root] = left_root
    for left in range(count):
        for right in range(left + 1, count):
            if distances[left, right] <= 1.0:
                union(left, right)

    return pd.Series({
        'composition_effective_decks': effective_decks,
        'one_change_archetypes': len({find(index) for index in range(count)}),
        'mean_nearest_card_changes': nearest.min(axis=1).mean() if count > 1 else 0.0,
        'mean_pairwise_card_changes': distances[np.triu_indices(count, 1)].mean() if count > 1 else 0.0,
    })

outer_summary = (
    genomes.groupby(['training_id', 'outer_gen'], as_index=False)
    .agg(
        population=('guid', 'nunique'),
        unique_decks=('signature', 'nunique'),
        mean_fitness=('score', 'mean'),
        median_fitness=('score', 'median'),
        fitness_std=('score', 'std'),
        best_fitness=('score', 'max'),
        missing_scores=('score', lambda values: values.isna().sum()),
    )
)
composition_rows = []
for (run_id, outer_gen), group in genomes.groupby(['training_id', 'outer_gen']):
    metrics = composition_diversity(group).to_dict()
    composition_rows.append({'training_id': run_id, 'outer_gen': outer_gen, **metrics})
composition_summary = pd.DataFrame(composition_rows)
outer_summary = outer_summary.merge(composition_summary, on=['training_id', 'outer_gen'])
outer_summary['exact_signature_ratio'] = outer_summary['unique_decks'] / outer_summary['population']
outer_summary['composition_diversity_ratio'] = outer_summary['composition_effective_decks'] / outer_summary['population']

species_rows = []
for row in run_catalog.itertuples(index=False):
    if row.training_id not in QUERY_IDS:
        continue
    for observed in row.metadata_dict.get('observed_populations', []):
        species_rows.append({'training_id': row.training_id, **observed})
species = pd.DataFrame(species_rows)

selected_outer = outer_summary[outer_summary.training_id.isin(EXPERIMENT_IDS)].copy()
selected_species = species[species.training_id.isin(EXPERIMENT_IDS)].copy() if not species.empty else species
single_outer_generation = selected_outer.outer_gen.nunique() == 1

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Best outer fitness', 'Fitness spread',
    'Composition-aware diversity', 'Species count'
))
if single_outer_generation:
    def add_run_distribution(frame, metric, row, col, color, label, value_format='.3f'):
        values = frame[metric].dropna()
        q25, median, q75 = values.quantile([0.25, 0.5, 0.75])
        fig.add_hrect(
            y0=q25, y1=q75, fillcolor=color, opacity=0.10,
            line_width=0, row=row, col=col,
        )
        fig.add_hline(
            y=median, line={'color': color, 'dash': 'dash', 'width': 1.5},
            row=row, col=col,
        )
        fig.add_trace(go.Scatter(
            x=frame.training_id, y=frame[metric], mode='markers',
            marker={'color': color, 'size': 9, 'line': {'color': 'white', 'width': 0.7}},
            customdata=frame[['population']].to_numpy() if 'population' in frame else None,
            hovertemplate=(
                'run %{x}<br>' + label + ': %{y:' + value_format + '}'
                + ('<br>population: %{customdata[0]}' if 'population' in frame else '')
                + '<extra></extra>'
            ),
            name=label, showlegend=False,
        ), row=row, col=col)
    add_run_distribution(selected_outer, 'best_fitness', 1, 1, '#3366CC', 'Best fitness', '.2f')
    add_run_distribution(selected_outer, 'fitness_std', 1, 2, '#DC3912', 'Fitness standard deviation', '.2f')
    add_run_distribution(selected_outer, 'composition_diversity_ratio', 2, 1, '#109618', 'Effective diversity ratio', '.3f')
    if not selected_species.empty:
        add_run_distribution(selected_species, 'species', 2, 2, '#FF9900', 'Species', '.0f')
    fig.update_xaxes(title_text='Training ID')
    chart_title = 'Independent Builder-Population Health (points = runs, band = IQR)'
else:
    for run_id, group in selected_outer.groupby('training_id'):
        fig.add_trace(go.Scatter(x=group.outer_gen, y=group.best_fitness, name=f'run {run_id}', legendgroup=str(run_id)), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.outer_gen, y=group.fitness_std, name=f'run {run_id}', legendgroup=str(run_id), showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=group.outer_gen, y=group.composition_diversity_ratio, name=f'run {run_id}', legendgroup=str(run_id), showlegend=False), row=2, col=1)
    for run_id, group in selected_species.groupby('training_id') if not selected_species.empty else []:
        fig.add_trace(go.Scatter(x=group.generation, y=group.species, name=f'run {run_id}', legendgroup=str(run_id), showlegend=False), row=2, col=2)
    fig.update_xaxes(title_text='Outer generation')
    chart_title = 'Outer Training Health'
fig.update_yaxes(title_text='Fitness', row=1, col=1)
fig.update_yaxes(title_text='Fitness standard deviation', row=1, col=2)
fig.update_yaxes(title_text='Similarity-adjusted effective decks / population', range=[0, 1.05], row=2, col=1)
fig.update_yaxes(title_text='Species', row=2, col=2)
fig.update_layout(height=750, title=chart_title, showlegend=not single_outer_generation)
fig.show()

best_generation = outer_summary.loc[
    outer_summary.groupby('training_id')['best_fitness'].idxmax(),
    ['training_id', 'outer_gen', 'best_fitness'],
].sort_values('training_id')
display(best_generation)
display(outer_summary[outer_summary.training_id.isin(EXPERIMENT_IDS)][[
    'training_id', 'outer_gen', 'population', 'fitness_std', 'unique_decks', 'exact_signature_ratio',
    'composition_effective_decks', 'composition_diversity_ratio',
    'one_change_archetypes', 'mean_nearest_card_changes', 'mean_pairwise_card_changes',
]])

## 4. Aggregated player-controller data

This is the expensive query. It reduces millions of raw games to one row per outer matchup, phase, and inner generation. The result is cached using the database state and selected run IDs; all later analyses reuse `pc_by_match`.

In [ ]:
incomplete_experiment_ids = tuple(run_catalog.loc[
    run_catalog.training_id.isin(EXPERIMENT_IDS)
    & ~run_catalog.termination_reason.fillna('').str.startswith('failed:')
    & (run_catalog.termination_reason != 'completed'),
    'training_id',
])
if incomplete_experiment_ids and not ALLOW_INCOMPLETE_EXPERIMENTS:
    raise RuntimeError(
        f'Experiment runs {incomplete_experiment_ids} are incomplete. '
        'Wait for training to finish before scanning pc_match, or explicitly set '
        'ALLOW_INCOMPLETE_EXPERIMENTS=True.'
    )

cache_state = f'v3:{TRAIN_DB.stat().st_size}:{TRAIN_DB.stat().st_mtime_ns}:{QUERY_IDS}'
cache_key = hashlib.sha256(cache_state.encode()).hexdigest()[:16]
cache_dir = PROJECT_ROOT / '.analysis_cache'
cache_path = cache_dir / f'pc_by_match_{cache_key}.pkl'

if USE_AGGREGATE_CACHE and cache_path.exists():
    pc_by_match = pd.read_pickle(cache_path)
    print(f'Loaded aggregate cache: {cache_path.name}')
else:
    pc_by_match = read_sql(f'''
    SELECT
        g.training_id,
        m.guid AS match_guid,
        m.gen AS outer_gen,
        m.deck_1,
        m.deck_2,
        COALESCE(p.phase, 'training') AS phase,
        p.gen AS inner_gen,
        COUNT(*) AS sample_count,
        COUNT(DISTINCT CASE WHEN COALESCE(p.phase, 'training') != 'training' THEN p.seed END) AS distinct_seeds,
        SUM(CASE WHEN p.seat_swap = 1 THEN 1 ELSE 0 END) AS swapped_games,
        SUM(CASE WHEN p.winner_id = 1 THEN 1 ELSE 0 END) AS p1_wins,
        SUM(CASE WHEN p.winner_id = 2 THEN 1 ELSE 0 END) AS p2_wins,
        SUM(CASE WHEN p.winner_id IS NULL AND p.p1_game_score IS NOT NULL THEN 1 ELSE 0 END) AS draws,
        SUM(CASE WHEN p.winner_id IS NOT NULL OR p.p1_game_score IS NOT NULL THEN 1 ELSE 0 END) AS outcome_games,
        SUM(CASE WHEN (p.seat_swap = 0 AND p.winner_id = 1)
                       OR (p.seat_swap = 1 AND p.winner_id = 2) THEN 1 ELSE 0 END) AS deck1_wins,
        SUM(CASE WHEN (p.seat_swap = 0 AND p.winner_id = 2)
                       OR (p.seat_swap = 1 AND p.winner_id = 1) THEN 1 ELSE 0 END) AS deck2_wins,
        AVG(p.rounds) AS average_rounds,
        AVG(p.g1_score) AS average_g1_score,
        AVG(p.g2_score) AS average_g2_score
    FROM pc_match p
    JOIN match_dbt m ON m.guid = p.match_dbt_guid
    JOIN genome g ON g.guid = m.genome_1
    WHERE g.training_id IN ({placeholders})
    GROUP BY g.training_id, m.guid, m.gen, m.deck_1, m.deck_2,
             COALESCE(p.phase, 'training'), p.gen
    ORDER BY g.training_id, m.gen, m.guid, phase, p.gen
    ''', query_ids)
    if USE_AGGREGATE_CACHE:
        cache_dir.mkdir(exist_ok=True)
        pc_by_match.to_pickle(cache_path)
        print(f'Saved aggregate cache: {cache_path.name}')

unexpected_pc_ids = tuple(sorted(set(pc_by_match.training_id) - set(QUERY_IDS)))
if unexpected_pc_ids:
    raise RuntimeError(f'Aggregate cache contains unselected training IDs: {unexpected_pc_ids}')
pc_by_match = pc_by_match[pc_by_match.training_id.isin(QUERY_IDS)].copy()
missing_pc_ids = tuple(sorted(set(EXPERIMENT_IDS) - set(pc_by_match.training_id)))
if missing_pc_ids:
    raise RuntimeError(f'Selected runs have no controller-game aggregates: {missing_pc_ids}')

print(f'{len(pc_by_match):,} aggregated rows from {pc_by_match.sample_count.sum():,.0f} raw games')
display(pc_by_match.groupby(['training_id', 'phase'], as_index=False).agg(
    matches=('match_guid', 'nunique'),
    generations=('inner_gen', 'nunique'),
    games=('sample_count', 'sum'),
))

In [ ]:
integrity_rows = []
catalog_by_id = run_catalog.set_index('training_id')
for run_id in EXPERIMENT_IDS:
    run_data = pc_by_match[pc_by_match.training_id == run_id]
    metadata_row = catalog_by_id.loc[run_id]
    expected_inner = metadata_row.configured_inner_generations
    expected_pairs = metadata_row.holdout_seed_pairs
    training = run_data[run_data.phase == 'training']
    mature = run_data[run_data.phase == 'holdout']
    early = run_data[run_data.phase == 'holdout_early']
    expected_holdout_games = 2 * expected_pairs if pd.notna(expected_pairs) else np.nan
    integrity_rows.append({
        'training_id': run_id,
        'termination': metadata_row.termination_reason,
        'outer_complete': metadata_row.observed_outer_generations == metadata_row.configured_outer_generations,
        'max_inner_generation': training.inner_gen.max() if not training.empty else np.nan,
        'inner_complete': training.inner_gen.max() == expected_inner if not training.empty else False,
        'mature_matches': mature.match_guid.nunique(),
        'mature_games_per_match_min': mature.groupby('match_guid').sample_count.sum().min() if not mature.empty else np.nan,
        'mature_games_per_match_max': mature.groupby('match_guid').sample_count.sum().max() if not mature.empty else np.nan,
        'expected_holdout_games': expected_holdout_games,
        'early_matches': early.match_guid.nunique(),
        'seat_balance_ok': bool((mature.groupby('match_guid').swapped_games.sum() * 2 == mature.groupby('match_guid').sample_count.sum()).all()) if not mature.empty else False,
        'null_outer_scores': int(genomes.loc[genomes.training_id == run_id, 'score'].isna().sum()),
        'git_dirty': metadata_row.git_dirty,
    })

integrity = pd.DataFrame(integrity_rows)
display(integrity)

failed = integrity[
    ~integrity['outer_complete']
    | ~integrity['inner_complete']
    | ~integrity['seat_balance_ok']
    | (integrity['null_outer_scores'] > 0)
]
if failed.empty:
    display(Markdown('**Integrity result:** all selected experiment runs passed structural checks.'))
else:
    display(Markdown('**Integrity result:** one or more runs failed. Exclude or investigate them before interpretation.'))
    display(failed)

## 5. Controller learning and player-experience proxies

Draw rate and game length describe behavioral stabilization, but cannot by themselves establish strategic competence.

In [ ]:
raw_training_rows = pc_by_match[pc_by_match.phase == 'training'].copy()
checkpoint_phases = {'holdout_early', 'holdout_checkpoint', 'holdout'}
checkpoint_rows = pc_by_match[pc_by_match.phase.isin(checkpoint_phases)].copy()
selected_with_raw = set(raw_training_rows.training_id) & set(EXPERIMENT_IDS)
if selected_with_raw == set(EXPERIMENT_IDS):
    training_rows = raw_training_rows
    curve_source_label = 'raw co-evolution games'
else:
    training_rows = checkpoint_rows
    curve_source_label = 'frozen, seat-balanced checkpoint holdouts'

controller_columns = [
    'training_id', 'inner_gen', 'games', 'draw_rate', 'p1_win_rate',
    'average_rounds', 'mean_g1_fitness', 'mean_g2_fitness',
]
controller_rows = []
for (run_id, inner_gen), group in training_rows.groupby(['training_id', 'inner_gen']):
    games = group.sample_count.sum()
    known_outcomes = group.outcome_games.sum()
    controller_rows.append({
        'training_id': run_id,
        'inner_gen': inner_gen,
        'games': games,
        'draw_rate': group.draws.sum() / games if known_outcomes else np.nan,
        'p1_win_rate': group.p1_wins.sum() / games if known_outcomes else np.nan,
        'average_rounds': weighted_mean(group, 'average_rounds'),
        'mean_g1_fitness': weighted_mean(group, 'average_g1_score'),
        'mean_g2_fitness': weighted_mean(group, 'average_g2_score'),
    })
controller_curve = pd.DataFrame(controller_rows, columns=controller_columns)
current_curve = controller_curve[controller_curve.training_id.isin(EXPERIMENT_IDS)].copy()

if current_curve.empty:
    display(Markdown('No controller training or checkpoint rows exist for the selected runs.'))
else:
    curve_summary = current_curve.groupby('inner_gen', as_index=False).agg(
        draw_median=('draw_rate', 'median'),
        draw_q25=('draw_rate', lambda values: values.quantile(0.25)),
        draw_q75=('draw_rate', lambda values: values.quantile(0.75)),
        rounds_median=('average_rounds', 'median'),
        rounds_q25=('average_rounds', lambda values: values.quantile(0.25)),
        rounds_q75=('average_rounds', lambda values: values.quantile(0.75)),
        fitness_median=('mean_g1_fitness', 'median'),
        fitness_q25=('mean_g1_fitness', lambda values: values.quantile(0.25)),
        fitness_q75=('mean_g1_fitness', lambda values: values.quantile(0.75)),
        runs=('training_id', 'nunique'),
    )
    fig = make_subplots(rows=1, cols=3, subplot_titles=(
        'Draw rate', 'Average rounds', 'Mean deck-1 fitness'
    ))
    if SHOW_INDIVIDUAL_RUNS:
        run_ids = sorted(current_curve.training_id.unique())
        run_colors = dict(zip(
            run_ids,
            px.colors.sample_colorscale('Turbo', [index / max(1, len(run_ids) - 1) for index in range(len(run_ids))]),
        ))
        for run_id, run_group in current_curve.groupby('training_id'):
            run_group = run_group.sort_values('inner_gen')
            for metric, col in (('draw_rate', 1), ('average_rounds', 2), ('mean_g1_fitness', 3)):
                fig.add_trace(go.Scatter(
                    x=run_group.inner_gen, y=run_group[metric],
                    mode='lines+markers',
                    line={'color': run_colors[run_id], 'width': 1},
                    marker={'size': 4}, opacity=0.32,
                    name=f'run {run_id}', legendgroup=str(run_id), showlegend=False,
                    hovertemplate=f'run {run_id}<br>generation %{{x}}<br>value %{{y:.3f}}<extra></extra>',
                ), row=1, col=col)
    for column, row, col, color in (
        ('draw', 1, 1, '#3366CC'),
        ('rounds', 1, 2, '#109618'),
        ('fitness', 1, 3, '#DC3912'),
    ):
        fig.add_trace(go.Scatter(
            x=curve_summary.inner_gen, y=curve_summary[f'{column}_q25'],
            mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip',
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=curve_summary.inner_gen, y=curve_summary[f'{column}_q75'],
            mode='lines', line={'width': 0}, fill='tonexty',
            fillcolor='rgba(80, 100, 130, 0.18)', showlegend=False,
            hoverinfo='skip',
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=curve_summary.inner_gen, y=curve_summary[f'{column}_median'],
            mode='lines+markers', line={'color': color, 'width': 2},
            name='Median across runs', showlegend=(col == 1),
        ), row=row, col=col)
    fig.update_xaxes(title_text='Inner generation')
    fig.update_yaxes(tickformat='.1%', title_text='Draw rate', row=1, col=1)
    fig.update_yaxes(title_text='Rounds', row=1, col=2)
    fig.update_yaxes(title_text='Fitness', row=1, col=3)
    fig.update_layout(
        height=450,
        title=f'Controller Checkpoint Curves ({curve_source_label})',
    )
    fig.show()
    display(curve_summary)

display(current_curve[current_curve.inner_gen.isin(CHECKPOINTS)].sort_values(['training_id', 'inner_gen']))

## 6. Active-ability learning proxy

For each card and inner generation, the coefficient below estimates the change in matchup fitness difference associated with one additional copy in deck 1 relative to deck 2. The regression uses one averaged observation per outer matchup, avoiding millions of controller-pair rows being treated as independent samples.

A coefficient becoming more positive with inner generation is consistent with controllers learning that card's playstyle. It is still observational: deck composition, opponents, and correlated cards can affect it.

In [ ]:
card_catalog = read_sql('''
    SELECT id AS card_id, name, hp, dmg, color_dmg_buff, color_hp_buff, ability, type
    FROM cardsdb.cards
    WHERE type = 'PLAYER'
    ORDER BY id
''')
active_ability_names = {'GoFirst', 'GoBefore1', 'AttackFront', 'HitAll1Dmg'}
card_catalog['ability_kind'] = np.where(
    card_catalog.ability.isin(active_ability_names), 'active',
    np.where(card_catalog.ability == 'NoAbility', 'none', 'passive'),
)
active_cards = tuple(card_catalog.loc[card_catalog.ability_kind == 'active', 'card_id'])
display(card_catalog)

copy_lookup = deck_cards.set_index(['deck_id', 'card_id']).copies
ability_rows = []
for card_id in active_cards:
    observations = training_rows[['training_id', 'inner_gen', 'match_guid', 'deck_1', 'deck_2', 'average_g1_score', 'average_g2_score']].copy()
    observations['deck1_copies'] = [copy_lookup.get((deck_id, card_id), 0) for deck_id in observations.deck_1]
    observations['deck2_copies'] = [copy_lookup.get((deck_id, card_id), 0) for deck_id in observations.deck_2]
    observations['copy_difference'] = observations.deck1_copies - observations.deck2_copies
    observations['fitness_difference'] = observations.average_g1_score - observations.average_g2_score
    observations = observations[observations.copy_difference != 0]
    for (run_id, inner_gen), group in observations.groupby(['training_id', 'inner_gen']):
        x = group.copy_difference.astype(float)
        y = group.fitness_difference.astype(float)
        denominator = ((x - x.mean()) ** 2).sum()
        coefficient = np.nan if denominator == 0 else ((x - x.mean()) * (y - y.mean())).sum() / denominator
        ability_rows.append({
            'training_id': run_id,
            'inner_gen': inner_gen,
            'card_id': card_id,
            'matchups': len(group),
            'fitness_per_copy': coefficient,
        })
ability_curve = pd.DataFrame(ability_rows, columns=[
    'training_id', 'inner_gen', 'card_id', 'matchups', 'fitness_per_copy'
])
selected_ability = ability_curve[ability_curve.training_id.isin(EXPERIMENT_IDS)].copy()
ability_summary = selected_ability.groupby(['card_id', 'inner_gen'], as_index=False).agg(
    fitness_median=('fitness_per_copy', 'median'),
    fitness_q25=('fitness_per_copy', lambda values: values.quantile(0.25)),
    fitness_q75=('fitness_per_copy', lambda values: values.quantile(0.75)),
    runs=('training_id', 'nunique'),
)

fig = make_subplots(rows=2, cols=2, subplot_titles=active_cards)
ability_colors = ('#3366CC', '#DC3912', '#109618', '#FF9900')
ability_run_ids = sorted(selected_ability.training_id.unique())
ability_run_colors = dict(zip(
    ability_run_ids,
    px.colors.sample_colorscale('Turbo', [index / max(1, len(ability_run_ids) - 1) for index in range(len(ability_run_ids))]),
))
for index, (card_id, group) in enumerate(ability_summary.groupby('card_id')):
    row, col = index // 2 + 1, index % 2 + 1
    if SHOW_INDIVIDUAL_RUNS:
        card_runs = selected_ability[selected_ability.card_id == card_id]
        for run_id, run_group in card_runs.groupby('training_id'):
            run_group = run_group.sort_values('inner_gen')
            fig.add_trace(go.Scatter(
                x=run_group.inner_gen, y=run_group.fitness_per_copy,
                mode='lines+markers',
                line={'color': ability_run_colors[run_id], 'width': 1},
                marker={'size': 3}, opacity=0.28, showlegend=False,
                hovertemplate=f'run {run_id}<br>generation %{{x}}<br>fitness/copy %{{y:.2f}}<extra></extra>',
            ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=group.inner_gen, y=group.fitness_q25, mode='lines',
        line={'width': 0}, showlegend=False, hoverinfo='skip',
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=group.inner_gen, y=group.fitness_q75, mode='lines',
        line={'width': 0}, fill='tonexty',
        fillcolor='rgba(80, 100, 130, 0.18)', showlegend=False, hoverinfo='skip',
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=group.inner_gen, y=group.fitness_median, mode='lines+markers',
        line={'color': ability_colors[index], 'width': 2},
        name=card_id, showlegend=False,
    ), row=row, col=col)
fig.add_hline(y=0, line_dash='dot', line_color='black')
fig.update_xaxes(title_text='Inner generation')
fig.update_yaxes(title_text='Fitness per extra copy')
fig.update_layout(height=680, title='Active-Card Value Proxy Across 40 Independent Runs')
fig.show()
display(ability_summary)

ability_checkpoints = ability_curve[ability_curve.inner_gen.isin(CHECKPOINTS)].pivot_table(
    index=['training_id', 'card_id'], columns='inner_gen', values='fitness_per_copy'
)
display(ability_checkpoints)

In [ ]:
stability_rows = []
for run_id, group in ability_curve.groupby('training_id'):
    final_gen = int(group.inner_gen.max())
    final_vector = group[group.inner_gen == final_gen].set_index('card_id').fitness_per_copy
    for inner_gen, generation_group in group.groupby('inner_gen'):
        vector = generation_group.set_index('card_id').fitness_per_copy.reindex(final_vector.index)
        valid = pd.concat([vector, final_vector], axis=1).dropna()
        rank_correlation = safe_corr(valid.iloc[:, 0].rank(), valid.iloc[:, 1].rank())
        mean_absolute_delta = (valid.iloc[:, 0] - valid.iloc[:, 1]).abs().mean()
        stability_rows.append({
            'training_id': run_id,
            'inner_gen': inner_gen,
            'final_gen': final_gen,
            'active_card_rank_correlation_to_final': rank_correlation,
            'active_card_mean_abs_delta_to_final': mean_absolute_delta,
        })
ability_stability = pd.DataFrame(stability_rows)

selected_stability = ability_stability[ability_stability.training_id.isin(EXPERIMENT_IDS)].copy()
stability_summary = selected_stability.groupby('inner_gen', as_index=False).agg(
    correlation_median=('active_card_rank_correlation_to_final', 'median'),
    correlation_q25=('active_card_rank_correlation_to_final', lambda values: values.quantile(0.25)),
    correlation_q75=('active_card_rank_correlation_to_final', lambda values: values.quantile(0.75)),
    mean_abs_delta=('active_card_mean_abs_delta_to_final', 'median'),
    runs=('training_id', 'nunique'),
)
fig = go.Figure()
if SHOW_INDIVIDUAL_RUNS:
    stability_run_ids = sorted(selected_stability.training_id.unique())
    stability_colors = dict(zip(
        stability_run_ids,
        px.colors.sample_colorscale('Turbo', [index / max(1, len(stability_run_ids) - 1) for index in range(len(stability_run_ids))]),
    ))
    for run_id, run_group in selected_stability.groupby('training_id'):
        run_group = run_group.sort_values('inner_gen')
        fig.add_trace(go.Scatter(
            x=run_group.inner_gen,
            y=run_group.active_card_rank_correlation_to_final,
            mode='lines+markers',
            line={'color': stability_colors[run_id], 'width': 1},
            marker={'size': 4}, opacity=0.30,
            name=f'run {run_id}', showlegend=False,
            hovertemplate=f'run {run_id}<br>generation %{{x}}<br>correlation %{{y:.3f}}<extra></extra>',
        ))
fig.add_trace(go.Scatter(
    x=stability_summary.inner_gen, y=stability_summary.correlation_q25,
    mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=stability_summary.inner_gen, y=stability_summary.correlation_q75,
    mode='lines', line={'width': 0}, fill='tonexty',
    fillcolor='rgba(80, 100, 130, 0.18)', name='Interquartile range',
    hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=stability_summary.inner_gen, y=stability_summary.correlation_median,
    mode='lines+markers', line={'color': '#3366CC', 'width': 2},
    name='Median across runs',
))
fig.update_yaxes(range=[-1.05, 1.05], title='Rank correlation to generation 50')
fig.update_xaxes(title='Inner generation')
fig.update_layout(title='Active-Card Ranking Agreement Across Independent Runs')
fig.add_hline(y=0.9, line_dash='dash', annotation_text='0.90 decision reference')
fig.show()
display(ability_stability[ability_stability.inner_gen.isin(CHECKPOINTS)].sort_values(['training_id', 'inner_gen']))

## 7. Replicated card-balance signals

Presence lift is calculated within each run's final outer generation. Treat cards with near-universal inclusion cautiously because their absence group is small.

In [ ]:
experiment_genomes = genomes[genomes.training_id.isin(EXPERIMENT_IDS)].copy()
final_outer = experiment_genomes.groupby('training_id').outer_gen.transform('max')
final_decks = experiment_genomes[experiment_genomes.outer_gen == final_outer].copy()

card_signal_rows = []
for run_id, run_decks in final_decks.groupby('training_id'):
    for card_id in card_catalog.card_id:
        copies = run_decks.deck_id.map(lambda deck_id: copy_lookup.get((deck_id, card_id), 0))
        present = copies > 0
        present_score = run_decks.loc[present, 'score'].mean()
        absent_score = run_decks.loc[~present, 'score'].mean()
        card_signal_rows.append({
            'training_id': run_id,
            'card_id': card_id,
            'decks': len(run_decks),
            'decks_present': int(present.sum()),
            'inclusion_rate': present.mean(),
            'average_copies_when_present': copies[present].mean(),
            'present_score': present_score,
            'absent_score': absent_score,
            'presence_lift': present_score - absent_score,
            'copy_score_correlation': safe_corr(copies, run_decks.score),
        })
card_signals = pd.DataFrame(card_signal_rows).merge(
    card_catalog[['card_id', 'name', 'ability', 'ability_kind']], on='card_id', how='left'
)

lift_matrix = card_signals.pivot(index='card_id', columns='training_id', values='presence_lift')
fig = px.imshow(
    lift_matrix,
    text_auto='.1f', aspect='auto', color_continuous_scale='RdBu', color_continuous_midpoint=0,
    title='Final-generation card presence lift by run',
    labels={'x': 'Training ID', 'y': 'Card', 'color': 'Presence lift'},
)
fig.update_layout(height=600)
fig.show()

display(card_signals.sort_values(['training_id', 'presence_lift'], ascending=[True, False]))
display(Markdown('**Cross-run Spearman rank agreement of card presence lift**'))
display(lift_matrix.rank().corr())

## 8. Held-out evaluation and player-experience sensitivity

In [ ]:
holdout = pc_by_match[pc_by_match.phase.isin(['holdout_early', 'holdout'])].copy()
holdout_rows = []
for (run_id, phase), group in holdout.groupby(['training_id', 'phase']):
    games = group.sample_count.sum()
    p1_wins = group.p1_wins.sum()
    p2_wins = group.p2_wins.sum()
    draws = group.draws.sum()
    low, high = wilson_interval(p1_wins, games)
    holdout_rows.append({
        'training_id': run_id,
        'phase': phase,
        'matches': group.match_guid.nunique(),
        'games': games,
        'p1_win_rate': p1_wins / games,
        'p1_win_ci_low': low,
        'p1_win_ci_high': high,
        'p2_win_rate': p2_wins / games,
        'draw_rate': draws / games,
        'average_rounds': weighted_mean(group, 'average_rounds'),
    })
holdout_summary = pd.DataFrame(holdout_rows)
display(holdout_summary)

perspectives = pd.concat([
    holdout[['training_id', 'match_guid', 'phase', 'deck_1', 'average_g1_score']].rename(
        columns={'deck_1': 'deck_id', 'average_g1_score': 'deck_score'}
    ),
    holdout[['training_id', 'match_guid', 'phase', 'deck_2', 'average_g2_score']].rename(
        columns={'deck_2': 'deck_id', 'average_g2_score': 'deck_score'}
    ),
], ignore_index=True)
early = perspectives[perspectives.phase == 'holdout_early'].rename(columns={'deck_score': 'early_score'})
mature = perspectives[perspectives.phase == 'holdout'].rename(columns={'deck_score': 'mature_score'})
phase_comparison = early.merge(mature, on=['training_id', 'match_guid', 'deck_id'])
phase_stat_rows = []
for run_id, group in phase_comparison.groupby('training_id'):
    phase_stat_rows.append({
        'training_id': run_id,
        'deck_observations': len(group),
        'early_mature_correlation': safe_corr(group.early_score, group.mature_score),
        'mean_absolute_change': (group.early_score - group.mature_score).abs().mean(),
    })
phase_stats = pd.DataFrame(phase_stat_rows)
display(phase_stats)

fig = px.scatter(
    phase_comparison, x='early_score', y='mature_score', color='training_id',
    opacity=0.45, trendline=None,
    title='Early versus mature held-out deck score',
)
bounds = [phase_comparison[['early_score', 'mature_score']].min().min(), phase_comparison[['early_score', 'mature_score']].max().max()]
fig.add_trace(go.Scatter(x=bounds, y=bounds, mode='lines', name='no change', line={'dash': 'dash', 'color': 'black'}))
fig.show()

## 9. Final-generation deck ranking with uncertainty

Intervals below describe held-out game uncertainty for each deck across its scheduled opponents. Games within a trained matchup are not fully independent, so use these intervals as descriptive rather than definitive inferential statistics.

In [ ]:
mature = pc_by_match[(pc_by_match.phase == 'holdout') & (pc_by_match.training_id.isin(EXPERIMENT_IDS))].copy()
deck_results = pd.concat([
    mature[['training_id', 'outer_gen', 'deck_1', 'deck1_wins', 'deck2_wins', 'draws', 'sample_count', 'average_g1_score']].rename(
        columns={'deck_1': 'deck_id', 'deck1_wins': 'wins', 'deck2_wins': 'losses', 'average_g1_score': 'fitness'}
    ),
    mature[['training_id', 'outer_gen', 'deck_2', 'deck2_wins', 'deck1_wins', 'draws', 'sample_count', 'average_g2_score']].rename(
        columns={'deck_2': 'deck_id', 'deck2_wins': 'wins', 'deck1_wins': 'losses', 'average_g2_score': 'fitness'}
    ),
], ignore_index=True)
deck_results['weighted_fitness'] = deck_results.fitness * deck_results.sample_count
deck_ranking = deck_results.groupby(['training_id', 'outer_gen', 'deck_id'], as_index=False).agg(
    wins=('wins', 'sum'), losses=('losses', 'sum'), draws=('draws', 'sum'),
    games=('sample_count', 'sum'), weighted_fitness=('weighted_fitness', 'sum'),
)
deck_ranking['win_rate'] = deck_ranking.wins / deck_ranking.games
deck_ranking['mean_fitness'] = deck_ranking.weighted_fitness / deck_ranking.games
intervals = deck_ranking.apply(lambda row: wilson_interval(row.wins, row.games), axis=1)
deck_ranking[['win_ci_low', 'win_ci_high']] = pd.DataFrame(intervals.tolist(), index=deck_ranking.index)

final_generation_map = genomes[genomes.training_id.isin(EXPERIMENT_IDS)].groupby('training_id').outer_gen.max()
deck_ranking = deck_ranking[deck_ranking.apply(lambda row: row.outer_gen == final_generation_map[row.training_id], axis=1)]
deck_ranking = deck_ranking.merge(
    genomes[['training_id', 'deck_id', 'signature', 'deck_name']].drop_duplicates(),
    on=['training_id', 'deck_id'], how='left',
)
deck_composition_rows = []
for deck_id, group in deck_cards.groupby('deck_id'):
    composition = ', '.join(f'{row.card_id} x{row.copies}' for row in group.sort_values('card_id').itertuples())
    deck_composition_rows.append({'deck_id': deck_id, 'composition': composition})
deck_compositions = pd.DataFrame(deck_composition_rows)
deck_ranking = deck_ranking.merge(deck_compositions, on='deck_id', how='left')
deck_ranking['rank'] = deck_ranking.groupby('training_id').mean_fitness.rank(method='dense', ascending=False).astype(int)
display(deck_ranking.sort_values(['training_id', 'rank']).groupby('training_id').head(10)[
    ['training_id', 'rank', 'deck_id', 'mean_fitness', 'win_rate', 'win_ci_low', 'win_ci_high', 'games', 'composition']
])

## 10. Inner-generation decision table

Choose the experienced-controller generation only after all four sensitivity seeds complete. The decision should consider active-card stability alongside outcome and game-length stability. A practical criterion is sustained active-card rank correlation of at least 0.90 with generation 35, without material movement in coefficients, draw rate, or rounds over the remaining generations.

In [ ]:
decision = ability_stability.merge(
    controller_curve[['training_id', 'inner_gen', 'draw_rate', 'average_rounds']],
    on=['training_id', 'inner_gen'], how='left',
)
decision = decision[decision.training_id.isin(EXPERIMENT_IDS) & decision.inner_gen.isin(CHECKPOINTS)]
display(decision.sort_values(['training_id', 'inner_gen']))

sensitivity_catalog = run_catalog[run_catalog.training_id.isin(discovered_sensitivity)]
complete_sensitivity = sensitivity_catalog[
    (sensitivity_catalog.termination_reason == 'completed')
    & (sensitivity_catalog.observed_outer_generations == sensitivity_catalog.configured_outer_generations)
]
if not discovered_sensitivity:
    display(Markdown(
        '**Decision status:** sensitivity experiments are outside the explicit run selection. '
        'This notebook view reports only the final collection.'
    ))
elif len(complete_sensitivity) < 4:
    display(Markdown(
        f'**Decision status:** {len(complete_sensitivity)}/4 complete `{SENSITIVITY_LABEL}` runs. '
        'Do not finalize the inner-generation recommendation yet.'
    ))
else:
    display(Markdown(
        '**Decision status:** all four sensitivity runs are complete. Review the decision table and '
        'require agreement across seeds before selecting the smallest defensible inner generation.'
    ))

## 11. Direct card-play and active-ability use

The inner-ceiling experiments record action summaries only in frozen holdouts. Cross-generation play tests whether later controllers directly outperform earlier controllers; this section remains empty for historical runs.

In [ ]:
action_summary = pd.DataFrame()
with db_connection() as conn:
    pc_columns = {row[1] for row in conn.execute('PRAGMA table_info(pc_match)')}
action_columns = {
    'p1_card_plays', 'p2_card_plays',
    'p1_active_ability_uses', 'p2_active_ability_uses',
    'deck1_controller_gen', 'deck2_controller_gen',
}

action_run_ids = tuple(EXPERIMENT_IDS)
if action_run_ids and action_columns.issubset(pc_columns):
    action_placeholders, action_ids = sql_in(action_run_ids)
    action_cache_state = f'v2:{TRAIN_DB.stat().st_size}:{TRAIN_DB.stat().st_mtime_ns}:{action_ids}'
    action_cache_key = hashlib.sha256(action_cache_state.encode()).hexdigest()[:16]
    action_cache_path = cache_dir / f'holdout_actions_{action_cache_key}.pkl'
    if USE_AGGREGATE_CACHE and action_cache_path.exists():
        action_games = pd.read_pickle(action_cache_path)
    else:
        action_games = read_sql(f'''
            SELECT
                g.training_id, m.guid AS match_guid, p.gen AS inner_gen,
                COALESCE(p.phase, 'training') AS phase,
                p.seat_swap, p.winner_id, p.g1_score, p.g2_score,
                p.deck1_controller_gen, p.deck2_controller_gen,
                p.p1_card_plays, p.p2_card_plays,
                p.p1_active_ability_uses, p.p2_active_ability_uses
            FROM pc_match p
            JOIN match_dbt m ON m.guid = p.match_dbt_guid
            JOIN genome g ON g.guid = m.genome_1
            WHERE g.training_id IN ({action_placeholders})
              AND p.p1_card_plays IS NOT NULL
              AND COALESCE(p.phase, 'training') != 'training'
        ''', action_ids)
        if USE_AGGREGATE_CACHE:
            cache_dir.mkdir(exist_ok=True)
            action_games.to_pickle(action_cache_path)

    action_rows = []
    for game in action_games.itertuples(index=False):
        for player_id in (1, 2):
            plays = json.loads(getattr(game, f'p{player_id}_card_plays') or '{}')
            uses = json.loads(getattr(game, f'p{player_id}_active_ability_uses') or '{}')
            won = int(game.winner_id == player_id)
            controls_deck1 = (player_id == 1) != bool(game.seat_swap)
            controller_gen = (
                game.deck1_controller_gen if controls_deck1
                else game.deck2_controller_gen
            )
            for card_id, play_count in plays.items():
                action_rows.append({
                    'training_id': game.training_id,
                    'inner_gen': game.inner_gen,
                    'controller_gen': controller_gen,
                    'phase': game.phase,
                    'card_id': card_id,
                    'plays': play_count,
                    'active_uses': uses.get(card_id, 0),
                    'player_games': 1,
                    'wins': won,
                })
    if action_rows:
        action_summary = pd.DataFrame(action_rows).groupby(
            ['training_id', 'inner_gen', 'controller_gen', 'phase', 'card_id'], as_index=False
        ).agg(
            plays=('plays', 'sum'), active_uses=('active_uses', 'sum'),
            player_games=('player_games', 'sum'), wins=('wins', 'sum'),
        )
        action_summary['active_use_rate_per_play'] = action_summary.active_uses / action_summary.plays
        action_summary['win_rate_when_played'] = action_summary.wins / action_summary.player_games
        active_action_summary = action_summary[
            action_summary.card_id.isin(active_cards)
            & (action_summary.phase != 'holdout_cross')
        ]
        active_action_across_runs = active_action_summary.groupby(
            ['controller_gen', 'card_id'], as_index=False
        ).agg(
            plays=('plays', 'sum'), active_uses=('active_uses', 'sum'),
            player_games=('player_games', 'sum'), wins=('wins', 'sum'),
            runs=('training_id', 'nunique'),
        )
        active_action_across_runs['active_use_rate_per_play'] = (
            active_action_across_runs.active_uses / active_action_across_runs.plays
        )
        active_action_across_runs['win_rate_when_played'] = (
            active_action_across_runs.wins / active_action_across_runs.player_games
        )
        fig = px.line(
            active_action_across_runs,
            x='controller_gen', y='active_use_rate_per_play',
            color='card_id', markers=True,
            title='Pooled Active-Ability Use Rate Across Selected Runs',
        )
        fig.update_yaxes(tickformat='.0%', range=[0, 1])
        fig.show()
        display(active_action_across_runs.sort_values(['card_id', 'controller_gen']))

    cross_games = action_games[action_games.phase == 'holdout_cross'].copy()
    if not cross_games.empty:
        cross_pairs = cross_games.groupby([
            'training_id', 'match_guid', 'deck1_controller_gen', 'deck2_controller_gen'
        ], as_index=False).agg(
            deck1_score=('g1_score', 'mean'), deck2_score=('g2_score', 'mean'),
            games=('match_guid', 'size'),
        )
        comparison_rows = []
        for run_id, run_pairs in cross_pairs.groupby('training_id'):
            final_generation = run_pairs[[
                'deck1_controller_gen', 'deck2_controller_gen'
            ]].max().max()
            reference_generation = run_pairs[[
                'deck1_controller_gen', 'deck2_controller_gen'
            ]].min().min()
            newer_deck1 = run_pairs[
                (run_pairs.deck1_controller_gen == final_generation)
                & (run_pairs.deck2_controller_gen == reference_generation)
            ]
            newer_deck2 = run_pairs[
                (run_pairs.deck1_controller_gen == reference_generation)
                & (run_pairs.deck2_controller_gen == final_generation)
            ]
            comparison = newer_deck1.merge(
                newer_deck2,
                on=['training_id', 'match_guid'],
                suffixes=('_newer_deck1', '_newer_deck2'),
            )
            comparison['newer_controller_advantage'] = (
                comparison.deck1_score_newer_deck1
                - comparison.deck1_score_newer_deck2
            )
            comparison['reference_gen'] = reference_generation
            comparison['final_gen'] = final_generation
            comparison_rows.append(comparison[[
                'training_id', 'match_guid', 'reference_gen', 'final_gen',
                'newer_controller_advantage',
            ]])
        cross_advantage = pd.concat(comparison_rows, ignore_index=True)
        display(Markdown('**Direct cross-generation play**'))
        cross_summary = cross_advantage.groupby([
            'reference_gen', 'final_gen', 'training_id'
        ]).newer_controller_advantage.agg(['count', 'mean', 'median', 'std'])
        cross_summary['standard_error'] = cross_summary['std'] / np.sqrt(cross_summary['count'])
        cross_summary['ci_low'] = cross_summary['mean'] - 1.96 * cross_summary['standard_error']
        cross_summary['ci_high'] = cross_summary['mean'] + 1.96 * cross_summary['standard_error']
        display(cross_summary)
else:
    display(Markdown('No direct holdout action data is available yet.'))

## Interpretation limits

- IDs 13–20 do not log card plays or ability uses. Their card-value trajectories cannot prove that an ability caused an outcome; direct action summaries begin with the `inner-ceiling-50` experiment.
- Presence lift and per-copy coefficients are observational and can be confounded by correlated deck composition.
- Legacy runs use a different controller representation, fitness function, evaluation protocol, and population behavior. They provide hypotheses, not direct replications.
- IDs 13–16 used 20 inner generations. The `post-replication` runs use 35 and must be reported as a separate sensitivity experiment.
- AI results estimate simulated competitive balance. Human enjoyment, comprehension, bluffing, fatigue, and creativity still require player evidence.
- The final thesis claim should be based on agreement with a documented tabletop calibration set, not comparison to memory alone.